# 🛡️ Detecção de Anomalias em Transações Financeiras
Este notebook demonstra a implementação de algoritmos de Machine Learning para identificação de fraudes e comportamentos atípicos em transações financeiras.

In [ ]:
# 1. Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Configuração de estilo dos gráficos
sns.set_theme(style="whitegrid")

## 📊 1. Geração / Carregamento do Dataset de Transações
Criando um conjunto de dados simulado contendo **transações normais** e uma pequena porcentagem de **anomalias/fraudes** (com valores discrepantes e horários atípicos).

In [ ]:
def generate_synthetic_data(num_records=2000):
    np.random.seed(42)
    
    # Transações normais (95% do total)
    amounts_normal = np.random.normal(loc=150, scale=50, size=int(num_records * 0.95))
    times_normal = np.random.uniform(6, 23, size=int(num_records * 0.95)) # Horário comercial/noite
    is_fraud_normal = np.zeros(int(num_records * 0.95))
    
    # Transações anômalas/fraudes (5% do total)
    amounts_fraud = np.random.normal(loc=3500, scale=800, size=int(num_records * 0.05))
    times_fraud = np.random.uniform(1, 4, size=int(num_records * 0.05)) # Madrugada
    is_fraud_fraud = np.ones(int(num_records * 0.05))
    
    df_normal = pd.DataFrame({'valor': amounts_normal, 'hora': times_normal, 'is_fraude': is_fraud_normal})
    df_fraud = pd.DataFrame({'valor': amounts_fraud, 'hora': times_fraud, 'is_fraude': is_fraud_fraud})
    
    df = pd.concat([df_normal, df_fraud]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df

df = generate_synthetic_data(2000)
print(f"Total de registros: {len(df)}")
print(f"Quantidade de fraudes: {int(df['is_fraude'].sum())}")
df.head()

## 🌲 2. Detecção Não Supervisionada (Isolation Forest)
O **Isolation Forest** isola observações selecionando aleatoriamente uma funcionalidade e um valor de divisão. Como as anomalias são poucas e diferentes, são isoladas mais rapidamente na árvore.

In [ ]:
features = ['valor', 'hora']
X = df[features]
y = df['is_fraude']

# Padronização das variáveis
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Treinamento do Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_iso'] = iso_forest.fit_predict(X_scaled)

# Converter saída: -1 (anomalia) vira 1, e 1 (normal) vira 0
df['pred_iso'] = df['anomaly_iso'].map({1: 0, -1: 1})

# Matriz de Confusão
print("Matriz de Confusão (Isolation Forest):")
print(confusion_matrix(y, df['pred_iso']))

## 🤖 3. Classificação Supervisionada (Random Forest)
Aplicação de um classificador **Random Forest** usando os rótulos de fraude para treinar o modelo e prever novas transações suspeitas.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Relatório de Classificação (Random Forest):")
print(classification_report(y_test, y_pred))

## 📈 4. Visualização dos Resultados
Gráfico de dispersão mostrando como o modelo identificou as anomalias com base em valor e horário.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hora', 
    y='valor', 
    hue='pred_iso', 
    palette={0: 'blue', 1: 'red'},
    style='is_fraude',
    alpha=0.8
)
plt.title("Detecção de Anomalias em Transações (Vermelho = Anomalia Identificada)")
plt.xlabel("Hora do Dia")
plt.ylabel("Valor da Transação (R$)")
plt.show()